In [ ]:
import glob
import pandas as pd
from autogluon.tabular import TabularDataset, TabularPredictor
from autogluon.common.features.feature_metadata import FeatureMetadata

In [ ]:
training_files = 5
data_files = glob.glob('bid_data/*.tsv')
dfs = [pd.read_csv(file, sep='\t') for file in data_files[:training_files]]

train_data = TabularDataset(pd.concat(dfs, ignore_index=True))

In [ ]:
train_data.head()

In [ ]:
label = 'points'
train_data[label].describe()

In [ ]:
exclude = []
outputs = ['num_tricks', 'points']
outputs.remove(label)

type_map = {k: 'int' for k in list(train_data)}
type_map['bid_pos'] = 'category'
type_map[label] = 'float'
metadata = FeatureMetadata(type_map_raw=type_map)

predictor = TabularPredictor(problem_type='regression', label=label)
predictor.fit(train_data.drop(columns=(exclude + outputs)), feature_metadata=metadata, presets='best_quality')

In [ ]:
test_data = TabularDataset(data_files[training_files])

y_pred = predictor.predict(test_data.drop(columns=([label] + exclude + outputs)))
y_pred.head()

In [ ]:
predictor.evaluate(test_data, silent=True)

In [ ]:
predictor.leaderboard(test_data)